# Phase 1 Check

In [2]:
import sys
import os

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# Imports
from src.data.ingestion import MarketDataIngestion
from src.data.spark_pipeline import get_spark_session, SparkTechnicalIndicators
from src.data.databricks_client import DatabricksClient

print("Imports successful!")

Imports successful!


In [3]:
# 1. Fetch OHLCV
fetcher = MarketDataIngestion()
pandas_df = fetcher.fetch_daily_ohlcv("AAPL")

# 2. Spark Transformations
spark = get_spark_session()
spark_df = spark.createDataFrame(pandas_df)

indicator_calc = SparkTechnicalIndicators(spark)
processed_df = indicator_calc.compute_indicators(spark_df)

# 3. Store Data
db_client = DatabricksClient()
db_client.write_dataset(processed_df, table_name="aapl_indicators")

print("Phase 1 Execution Complete!")

[2026-09-22 02:44:06] [INFO] [stonks_maker]: Fetching OHLCV for AAPL via yfinance...
[2026-09-22 02:44:13] [INFO] [stonks_maker]: Starting Spark indicator transformations...
[2026-09-22 02:44:14] [INFO] [stonks_maker]: Completed Spark indicator calculations.
[2026-09-22 02:44:14] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.
[2026-09-22 02:44:17] [INFO] [stonks_maker]: Successfully saved to data/processed/aapl_indicators
Phase 1 Execution Complete!


In [5]:
# Verify saved Parquet/Delta dataset
output_df = db_client.read_dataset(spark, table_name="aapl_indicators")
output_df.show(5)

[2026-09-22 03:13:33] [INFO] [stonks_maker]: Reading dataset locally from data/processed/aapl_indicators
+----------+------+------------------+------------------+------------------+------------------+---------+------------------+------------------+------------------+------------------+------------------+
|      date|ticker|              open|              high|               low|             close|   volume|            sma_20|            sma_50|   bollinger_upper|   bollinger_lower|            rsi_14|
+----------+------+------------------+------------------+------------------+------------------+---------+------------------+------------------+------------------+------------------+------------------+
|2025-09-22|  AAPL|247.38616206443075|255.69547916115815| 247.2068166293636|255.13751220703125|105517400|255.13751220703125|255.13751220703125|              NULL|              NULL|               0.0|
|2025-09-23|  AAPL|254.93828225227924| 256.3929004583055|252.64674396870498| 253.4936065673

# Phase 2 Check